# Projeto de Visão Computacional - YOLO
## Hugo Mariano - RM560688
### Fase 6 - PBL | FarmTech Solutions

Este notebook documenta o desenvolvimento completo do sistema de visão computacional com YOLO customizado, conforme o planejamento do projeto e as tasks das fases 3 e 4.

## Objetivo
- Detectar e classificar dois objetos distintos usando YOLO customizado
- Comparar desempenho entre diferentes configurações de treinamento
- Testar o modelo em imagens separadas e analisar resultados


## 1. Importação

Importamos todas as bibliotecas necessárias para manipulação de dados, visualização e avaliação dos modelos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
import tensorflow as tf
import sklearn
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import os
import glob
import random
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report
from matplotlib.ticker import MaxNLocator
from PIL import Image
import time

## 2. Configuração dos Caminhos e Parâmetros

Configuramos os caminhos dos dados, parâmetros de treino e as classes do projeto. Utilizamos o arquivo `data.yaml` já existente no repositório para configurar o dataset e as classes.

In [ ]:
# Caminhos principais
train_dir = '../dataset/train/images'
val_dir = '../dataset/val/images'
test_dir = '../dataset/test/images'
labels_dir = '../dataset/train/labels'
models_dir = '../models/'
results_dir = '../results/yolo_custom/'
data_yaml = '../config/data.yaml'

# Hiperparâmetros
epochs_1 = 30
epochs_2 = 60
batch_size = 16
learning_rate = 0.001
imgsz = 640

# Criação dos diretórios de saída, se não existirem
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Verificamos se os diretórios de dados existem
for path in [train_dir, val_dir, test_dir]:
    if not os.path.exists(path):
        print(f"AVISO: O caminho {path} não existe!")

# Definimos as classes do projeto (apenas para visualização, as classes reais estão no data.yaml)
classes = ['copo', 'headphones']

## 3. Visualização de Amostras do Dataset

Visualizamos exemplos das imagens para garantir que o dataset está correto.

In [ ]:
def show_images_from_folder(folder, n=4):
    imgs = glob.glob(os.path.join(folder, '*.jpg')) + glob.glob(os.path.join(folder, '*.png'))
    if not imgs:
        print(f"Nenhuma imagem encontrada em {folder}")
        return
    random.shuffle(imgs)
    plt.figure(figsize=(15, 4))
    for i, img_path in enumerate(imgs[:n]):
        img = Image.open(img_path)
        plt.subplot(1, n, i+1)
        plt.imshow(img)
        plt.title(f"{os.path.basename(img_path)}\n{img.size}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f"Total de imagens: {len(imgs)}")

print('Amostras de treino:')
show_images_from_folder(train_dir)
print('Amostras de validação:')
show_images_from_folder(val_dir)
print('Amostras de teste:')
show_images_from_folder(test_dir)

## 4. Treinamento do Modelo YOLO Customizado

Treinamos dois modelos YOLO com diferentes números de épocas para comparar desempenho.

In [ ]:

# Treinamento com 30 épocas
print("Iniciando treinamento com 30 épocas...")
model = YOLO('yolov8n.pt')  # Modelo base
results_30 = model.train(
    data=data_yaml, 
    epochs=epochs_1, 
    batch=batch_size, 
    imgsz=imgsz, 
    project=models_dir, 
    name='yolo_30ep',
    patience=15,  # Early stopping
    verbose=True
)
model.save(os.path.join(models_dir, 'model_30epochs.pt'))
print(f"Modelo de 30 épocas salvo em {os.path.join(models_dir, 'model_30epochs.pt')}")

# Treinamento com 60 épocas
print("Iniciando treinamento com 60 épocas...")
results_60 = model.train(
    data=data_yaml, 
    epochs=epochs_2, 
    batch=batch_size, 
    imgsz=imgsz, 
    project=models_dir, 
    name='yolo_60ep',
    patience=20,  # Early stopping
    verbose=True
)
model.save(os.path.join(models_dir, 'model_60epochs.pt'))
print(f"Modelo de 60 épocas salvo em {os.path.join(models_dir, 'model_60epochs.pt')}")

## 5. Avaliação dos Modelos

# Análise de Resultados: Modelos YOLO Customizados


##  Diagnóstico Geral

Após a avaliação detalhada dos resultados obtidos com os dois modelos YOLO, constatamos uma clara superioridade do modelo treinado com 60 épocas em todos os aspectos de desempenho. Observamos melhorias substanciais nas métricas-chave, com destaque para o aumento de 42% no recall e 28% no F1-Score, indicando um sistema muito mais confiável e preciso para detecção de objetos.

##  Análise das Curvas de Aprendizado

### Perdas de Treinamento e Validação

Analisando os gráficos de Box Loss e Class Loss, observamos:

- O modelo de 60 épocas atingiu valores significativamente menores de perdas ao final do treinamento (0.42 para Box Loss)
- Ambos os modelos apresentaram oscilações nas perdas de validação, porém o modelo de 60 épocas demonstrou maior estabilidade nas épocas finais
- As oscilações observadas são esperadas considerando o tamanho limitado do nosso dataset (80 imagens)

## Métricas de Desempenho

Comparando os melhores resultados de cada modelo:

| Métrica | Modelo 30 épocas | Modelo 60 épocas | Melhoria |
|---------|------------------|------------------|----------|
| Precisão | 0.847 | 0.966 | +14.11% |
| Recall | 0.700 | 0.994 | +42.01% |
| mAP50 | 0.859 | 0.995 | +15.85% |
| mAP50-95 | 0.658 | 0.757 | +14.91% |
| F1-Score | 0.767 | 0.980 | +27.87% |

Destacamos:

- A melhoria significativa no recall (42%) indica que o modelo de 60 épocas detecta praticamente todos os objetos presentes nas imagens
- O modelo de 60 épocas atingiu níveis de precisão e mAP50 próximos a 1.0, demonstrando detecção quase perfeita
- O F1-Score de 0.98 evidencia um excelente balanceamento entre precisão e recall

Os gráficos de evolução dessas métricas demonstram que o modelo de 60 épocas não apenas alcançou valores superiores, mas também apresentou maior estabilidade a partir da época 45.

## Convergência e Eficiência

Nossos dados revelam que:

- O modelo de 30 épocas atingiu seu melhor desempenho na última época (30), sugerindo que poderia se beneficiar de treinamento adicional
- O modelo de 60 épocas obteve seus melhores resultados na época 52
- O F1-Score do modelo de 60 épocas estabilizou em valores superiores a 0.98 a partir da época 45
- O custo computacional para o modelo de 60 épocas foi aproximadamente 4 vezes maior (25,37 horas vs 6,55 horas)

## Conclusões

Com base nos resultados, concluímos que:

**Desempenho superior**: O modelo de 60 épocas oferece melhorias significativas em todas as métricas avaliadas, justificando seu uso para aplicações onde precisão e recall são críticos.

 **Otimização de recursos**: Identificamos que um mecanismo de early stopping mais sensível poderia ter interrompido o treinamento por volta da época 45, economizando recursos computacionais sem comprometer significativamente o desempenho.

**Estabilidade**: A maior estabilidade das métricas no modelo com mais épocas sugere maior robustez e confiabilidade para aplicações em produção.

 **Impacto prático**: O recall próximo a 100% (0.994) representa uma vantagem competitiva concreta para aplicações de segurança e controle de acesso da FarmTech Solutions, onde a não detecção de objetos (falsos negativos) pode ter consequências significativas.

Para implementação no ambiente da FarmTech Solutions, recomendamos o modelo treinado com 60 épocas para demonstrações ao cliente e aplicações críticas, destacando particularmente os valores excepcionais de precisão e recall obtidos.

In [ ]:

# Carregar os dados dos CSVs
results_30ep = pd.read_csv('../models/yolo_30ep/results.csv')
results_60ep = pd.read_csv('../models/yolo_60ep/results.csv')

# Adicionar coluna indicando o modelo
results_30ep['modelo'] = '30 épocas'
results_60ep['modelo'] = '60 épocas'


plt.style.use('ggplot')
sns.set_palette('colorblind')

# ----- 1. Comparação das curvas de aprendizado (losses) -----
plt.figure(figsize=(15, 10))

# Losses de treinamento
plt.subplot(2, 2, 1)
plt.plot(results_30ep['epoch'], results_30ep['train/box_loss'], label='30ep - Box Loss', marker='o', markersize=3, alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['train/box_loss'], label='60ep - Box Loss', marker='s', markersize=3, alpha=0.7)
plt.title('Comparação: Box Loss (Treinamento)', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(2, 2, 2)
plt.plot(results_30ep['epoch'], results_30ep['train/cls_loss'], label='30ep - Class Loss', marker='o', markersize=3, alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['train/cls_loss'], label='60ep - Class Loss', marker='s', markersize=3, alpha=0.7)
plt.title('Comparação: Class Loss (Treinamento)', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Losses de validação
plt.subplot(2, 2, 3)
plt.plot(results_30ep['epoch'], results_30ep['val/box_loss'], label='30ep - Val Box Loss', marker='o', markersize=3, alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['val/box_loss'], label='60ep - Val Box Loss', marker='s', markersize=3, alpha=0.7)
plt.title('Comparação: Box Loss (Validação)', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(2, 2, 4)
plt.plot(results_30ep['epoch'], results_30ep['val/cls_loss'], label='30ep - Val Class Loss', marker='o', markersize=3, alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['val/cls_loss'], label='60ep - Val Class Loss', marker='s', markersize=3, alpha=0.7)
plt.title('Comparação: Class Loss (Validação)', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('comparacao_losses.png', dpi=300, bbox_inches='tight')
plt.show()

# ----- 2. Comparação das métricas de desempenho -----
plt.figure(figsize=(15, 10))

# Precisão
plt.subplot(2, 2, 1)
plt.plot(results_30ep['epoch'], results_30ep['metrics/precision(B)'], label='30 épocas', marker='o', markersize=3, color='blue', alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['metrics/precision(B)'], label='60 épocas', marker='s', markersize=3, color='green', alpha=0.7)
plt.title('Evolução da Precisão', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Precisão')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Recall
plt.subplot(2, 2, 2)
plt.plot(results_30ep['epoch'], results_30ep['metrics/recall(B)'], label='30 épocas', marker='o', markersize=3, color='blue', alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['metrics/recall(B)'], label='60 épocas', marker='s', markersize=3, color='green', alpha=0.7)
plt.title('Evolução do Recall', fontsize=13)
plt.xlabel('Época')
plt.ylabel('Recall')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# mAP50
plt.subplot(2, 2, 3)
plt.plot(results_30ep['epoch'], results_30ep['metrics/mAP50(B)'], label='30 épocas', marker='o', markersize=3, color='blue', alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['metrics/mAP50(B)'], label='60 épocas', marker='s', markersize=3, color='green', alpha=0.7)
plt.title('Evolução do mAP50', fontsize=13)
plt.xlabel('Época')
plt.ylabel('mAP50')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# mAP50-95
plt.subplot(2, 2, 4)
plt.plot(results_30ep['epoch'], results_30ep['metrics/mAP50-95(B)'], label='30 épocas', marker='o', markersize=3, color='blue', alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['metrics/mAP50-95(B)'], label='60 épocas', marker='s', markersize=3, color='green', alpha=0.7)
plt.title('Evolução do mAP50-95', fontsize=13)
plt.xlabel('Época')
plt.ylabel('mAP50-95')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('comparacao_metricas.png', dpi=300, bbox_inches='tight')
plt.show()

# ----- 3. Calcular e plotar F1-Score para ambos os modelos -----
f1_30ep = 2 * (results_30ep['metrics/precision(B)'] * results_30ep['metrics/recall(B)']) / (
    results_30ep['metrics/precision(B)'] + results_30ep['metrics/recall(B)'])

f1_60ep = 2 * (results_60ep['metrics/precision(B)'] * results_60ep['metrics/recall(B)']) / (
    results_60ep['metrics/precision(B)'] + results_60ep['metrics/recall(B)'])

plt.figure(figsize=(12, 6))
plt.plot(results_30ep['epoch'], f1_30ep, label='30 épocas', marker='o', markersize=3, color='blue', alpha=0.7)
plt.plot(results_60ep['epoch'], f1_60ep, label='60 épocas', marker='s', markersize=3, color='green', alpha=0.7)
plt.title('Evolução do F1-Score', fontsize=14)
plt.xlabel('Época')
plt.ylabel('F1-Score')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('comparacao_f1.png', dpi=300, bbox_inches='tight')
plt.show()

# ----- 4. Análise dos melhores resultados de cada modelo -----
# Encontrar a melhor época para cada modelo (baseado no mAP50-95)
best_30ep = results_30ep.loc[results_30ep['metrics/mAP50-95(B)'].idxmax()]
best_60ep = results_60ep.loc[results_60ep['metrics/mAP50-95(B)'].idxmax()]

print("\n===== COMPARAÇÃO DOS MELHORES RESULTADOS =====")
print(f"Modelo de 30 épocas - Melhor resultado na época {int(best_30ep['epoch'])}:")
print(f"- Precisão: {best_30ep['metrics/precision(B)']:.4f}")
print(f"- Recall: {best_30ep['metrics/recall(B)']:.4f}")
print(f"- mAP50: {best_30ep['metrics/mAP50(B)']:.4f}")
print(f"- mAP50-95: {best_30ep['metrics/mAP50-95(B)']:.4f}")
print(f"- F1-Score: {f1_30ep[best_30ep.name]:.4f}")

print(f"\nModelo de 60 épocas - Melhor resultado na época {int(best_60ep['epoch'])}:")
print(f"- Precisão: {best_60ep['metrics/precision(B)']:.4f}")
print(f"- Recall: {best_60ep['metrics/recall(B)']:.4f}")
print(f"- mAP50: {best_60ep['metrics/mAP50(B)']:.4f}")
print(f"- mAP50-95: {best_60ep['metrics/mAP50-95(B)']:.4f}")
print(f"- F1-Score: {f1_60ep[best_60ep.name]:.4f}")

# Calcular percentual de melhoria
melhoria = {
    'Precisão': (best_60ep['metrics/precision(B)'] - best_30ep['metrics/precision(B)']) / best_30ep['metrics/precision(B)'] * 100,
    'Recall': (best_60ep['metrics/recall(B)'] - best_30ep['metrics/recall(B)']) / best_30ep['metrics/recall(B)'] * 100,
    'mAP50': (best_60ep['metrics/mAP50(B)'] - best_30ep['metrics/mAP50(B)']) / best_30ep['metrics/mAP50(B)'] * 100,
    'mAP50-95': (best_60ep['metrics/mAP50-95(B)'] - best_30ep['metrics/mAP50-95(B)']) / best_30ep['metrics/mAP50-95(B)'] * 100,
    'F1-Score': (f1_60ep[best_60ep.name] - f1_30ep[best_30ep.name]) / f1_30ep[best_30ep.name] * 100
}

print("\n===== PERCENTUAL DE MELHORIA (60ep vs 30ep) =====")
for metrica, valor in melhoria.items():
    print(f"- {metrica}: {valor:.2f}%")

# ----- 5. Visualização dos melhores resultados em um gráfico de barras -----
# Preparar dados para o gráfico
melhores_resultados = pd.DataFrame({
    '30 épocas': [
        best_30ep['metrics/precision(B)'], 
        best_30ep['metrics/recall(B)'], 
        best_30ep['metrics/mAP50(B)'], 
        best_30ep['metrics/mAP50-95(B)'],
        f1_30ep[best_30ep.name]
    ],
    '60 épocas': [
        best_60ep['metrics/precision(B)'], 
        best_60ep['metrics/recall(B)'], 
        best_60ep['metrics/mAP50(B)'], 
        best_60ep['metrics/mAP50-95(B)'],
        f1_60ep[best_60ep.name]
    ]
}, index=['Precisão', 'Recall', 'mAP50', 'mAP50-95', 'F1-Score'])

# Criar gráfico de barras comparativas
plt.figure(figsize=(14, 8))
x = np.arange(len(melhores_resultados.index))
width = 0.35

plt.bar(x - width/2, melhores_resultados['30 épocas'], width, label='30 épocas', color='royalblue')
plt.bar(x + width/2, melhores_resultados['60 épocas'], width, label='60 épocas', color='forestgreen')

# Adicionar labels, título e outros elementos
plt.xlabel('Métrica', fontsize=12)
plt.ylabel('Valor', fontsize=12)
plt.title('Comparação dos Melhores Resultados', fontsize=15)
plt.xticks(x, melhores_resultados.index, fontsize=11)
plt.ylim(0, 1.1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(fontsize=11)

# Adicionar valores nas barras
for i, v1 in enumerate(melhores_resultados['30 épocas']):
    plt.text(i - width/2, v1 + 0.02, f'{v1:.3f}', ha='center', fontsize=10, fontweight='bold')
    
for i, v2 in enumerate(melhores_resultados['60 épocas']):
    plt.text(i + width/2, v2 + 0.02, f'{v2:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('melhores_resultados.png', dpi=300, bbox_inches='tight')
plt.show()

# ----- 6. Análise do tempo de treinamento -----
tempos_30ep = results_30ep['time'].sum()
tempos_60ep = results_60ep['time'].sum()

print(f"\nTempo total de treinamento (30 épocas): {tempos_30ep:.2f} segundos ({tempos_30ep/60:.2f} minutos)")
print(f"Tempo total de treinamento (60 épocas): {tempos_60ep:.2f} segundos ({tempos_60ep/60:.2f} minutos)")
print(f"Tempo médio por época (30 épocas): {results_30ep['time'].mean():.2f} segundos")
print(f"Tempo médio por época (60 épocas): {results_60ep['time'].mean():.2f} segundos")

# Gráfico comparativo de tempo por época
plt.figure(figsize=(12, 6))
plt.plot(results_30ep['epoch'], results_30ep['time'], label='30 épocas', marker='o', markersize=4, alpha=0.7)
plt.plot(results_60ep['epoch'], results_60ep['time'], label='60 épocas', marker='s', markersize=4, alpha=0.7)
plt.title('Tempo por Época', fontsize=14)
plt.xlabel('Época')
plt.ylabel('Tempo (segundos)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('comparacao_tempos.png', dpi=300, bbox_inches='tight')
plt.show()

# ----- 7. Resumo de convergência e eficiência -----
print("\n===== ANÁLISE DE CONVERGÊNCIA =====")
# Calcular a taxa de convergência (quanto tempo levou para atingir 95% do melhor resultado)
# Para mAP50-95
map_30ep_max = results_30ep['metrics/mAP50-95(B)'].max()
map_60ep_max = results_60ep['metrics/mAP50-95(B)'].max()

convergence_30ep = results_30ep[results_30ep['metrics/mAP50-95(B)'] >= 0.95 * map_30ep_max]['epoch'].min()
convergence_60ep = results_60ep[results_60ep['metrics/mAP50-95(B)'] >= 0.95 * map_60ep_max]['epoch'].min()

print(f"O modelo de 30 épocas atingiu 95% do seu melhor mAP50-95 na época {convergence_30ep}")
print(f"O modelo de 60 épocas atingiu 95% do seu melhor mAP50-95 na época {convergence_60ep}")

# Calcular a eficiência (tempo até convergência)
time_to_conv_30ep = results_30ep[results_30ep['epoch'] <= convergence_30ep]['time'].sum()
time_to_conv_60ep = results_60ep[results_60ep['epoch'] <= convergence_60ep]['time'].sum()

print(f"Tempo até convergência (30 épocas): {time_to_conv_30ep:.2f} segundos ({time_to_conv_30ep/60:.2f} minutos)")
print(f"Tempo até convergência (60 épocas): {time_to_conv_60ep:.2f} segundos ({time_to_conv_60ep/60:.2f} minutos)")

# Calcular eficiência (melhoria por tempo gasto)
efficiency_30ep = map_30ep_max / tempos_30ep
efficiency_60ep = map_60ep_max / tempos_60ep

print(f"Eficiência (mAP por segundo) - 30 épocas: {efficiency_30ep:.6f}")
print(f"Eficiência (mAP por segundo) - 60 épocas: {efficiency_60ep:.6f}")
print(f"Melhoria de eficiência: {(efficiency_60ep - efficiency_30ep) / efficiency_30ep * 100:.2f}%")

## 6. Análise Qualitativa das Detecções dos Modelos YOLO

## Observações sobre as Detecções

Analisamos qualitativamente as detecções realizadas pelos dois modelos YOLO (30 e 60 épocas) nas imagens de teste, e destacamos algumas observações importantes que complementam nossa análise quantitativa anterior.

### Detecção da classe "copo" (Imagem 1)

Na primeira imagem, observamos:

- **Precisão da classificação**: Ambos os modelos identificaram corretamente o objeto como "copo", com confiança muito alta (0.93 para o modelo de 30 épocas e 0.99 para o modelo de 60 épocas).

- **Qualidade da bounding box**: Os dois modelos geraram bounding boxes muito similares, com excelente ajuste ao contorno do objeto.

- **Diferença de confiança**: O modelo de 60 épocas apresentou uma confiança 6% superior ao modelo de 30 épocas, mesmo em um caso relativamente simples.

- **Precisão da detecção**: Este caso demonstra que ambos os modelos são altamente eficazes em detectar objetos com características visuais claras e bem definidas.

### Detecção da classe "headphones" (Imagem 2)

Na segunda imagem, observamos:

- **Detecção de objetos desafiadores**: Ambos os modelos foram capazes de detectar corretamente os fones de ouvido, mesmo com sua forma complexa e coloração escura que pode dificultar a identificação.

- **Diferença de confiança interessante**: Neste caso, o modelo de 30 épocas apresentou uma confiança ligeiramente superior (0.92) em comparação ao modelo de 60 épocas (0.83).

- **Qualidade da bounding box**: A delimitação do objeto é bastante similar entre os dois modelos, abrangendo adequadamente toda a extensão dos fones.

- **Possível especialização**: Este resultado sugere que o modelo de 30 épocas pode ter desenvolvido uma especialização particular para a classe "headphones" durante seu treinamento.

## Implicações dos Resultados

Os resultados visuais complementam nossa análise quantitativa e nos levam a algumas conclusões adicionais:

1. **Robustez dos modelos**: Ambos os modelos demonstram bom desempenho nas imagens de teste para as duas classes, indicando boa generalização do aprendizado.

2. **Variação por classe**: O desempenho relativo entre os modelos pode variar dependendo da classe específica, sugerindo que o aumento do número de épocas não beneficia uniformemente todas as classes.

3. **Alta confiança geral**: Os valores de confiança elevados (todos acima de 0.80) indicam que o dataset de treinamento, apesar de relativamente pequeno (aproximadamente 80 imagens), proporcionou uma base adequada para o aprendizado das características distintivas de cada classe.

4. **Aplicabilidade prática**: A qualidade das detecções confirma a viabilidade de implementação desses modelos em aplicações reais da FarmTech Solutions, como controle de acesso baseado no reconhecimento de objetos.

## Considerações Finais

A análise qualitativa reforça que, apesar do modelo de 60 épocas apresentar métricas quantitativas gerais superiores, ambos os modelos alcançaram um nível de desempenho que pode ser considerado adequado para aplicações práticas. A escolha entre eles deve considerar não apenas as métricas agregadas, mas também requisitos específicos da aplicação, como velocidade de inferência e exigências de precisão para cada classe de objeto.


In [ ]:


# Função para mostrar previsões lado a lado para ambos os modelos
def comparar_previsoes(img_path, modelo_30ep, modelo_60ep, conf=0.25):
    """
    Compara as previsões dos dois modelos na mesma imagem.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Imagem original
    img = Image.open(img_path)
    axes[0].imshow(img)
    axes[0].set_title('Imagem Original', fontsize=12)
    axes[0].axis('off')
    
    # Previsão do modelo de 30 épocas
    results_30 = modelo_30ep(img_path, conf=conf)
    axes[1].imshow(results_30[0].plot())
    axes[1].set_title('Modelo 30 épocas', fontsize=12)
    axes[1].axis('off')
    
    # Previsão do modelo de 60 épocas
    results_60 = modelo_60ep(img_path, conf=conf)
    axes[2].imshow(results_60[0].plot())
    axes[2].set_title('Modelo 60 épocas', fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Imprimir detalhes das detecções
    print(f"Imagem: {os.path.basename(img_path)}")
    
    print("\nDetecções - Modelo 30 épocas:")
    for i, box in enumerate(results_30[0].boxes):
        classe = classes[int(box.cls)]
        confianca = float(box.conf)
        print(f"  - Objeto {i+1}: Classe '{classe}', Confiança: {confianca:.4f}")
    
    print("\nDetecções - Modelo 60 épocas:")
    for i, box in enumerate(results_60[0].boxes):
        classe = classes[int(box.cls)]
        confianca = float(box.conf)
        print(f"  - Objeto {i+1}: Classe '{classe}', Confiança: {confianca:.4f}")


# Carregar os modelos treinados
modelo_30ep = YOLO(os.path.join(models_dir, 'model_30epochs.pt'))
modelo_60ep = YOLO(os.path.join(models_dir, 'model_60epochs.pt'))

# Selecionar imagens de teste
test_imgs = glob.glob(os.path.join(test_dir, '*.jpg')) + glob.glob(os.path.join(test_dir, '*.png'))

if len(test_imgs) == 0:
    print("Nenhuma imagem de teste encontrada no diretório especificado!")
else:
    print(f"Total de imagens de teste disponíveis: {len(test_imgs)}")
    
    # Separar imagens por classe (se possível)
    # Primeiro, precisamos prever a classe de cada imagem usando o modelo mais confiável (60 épocas)
    print("Classificando imagens de teste...")
    imgs_copo = []
    imgs_fone = []
    
    for img_path in test_imgs:
        result = modelo_60ep(img_path, conf=0.5)
        if len(result[0].boxes) > 0:
            # Pegar a classe da primeira detecção (com maior confiança)
            classe = int(result[0].boxes[0].cls)
            if classe == 0:  # assumindo que 'copo' é a classe 0
                imgs_copo.append(img_path)
            else:
                imgs_fone.append(img_path)
    
    print(f"Encontrados: {len(imgs_copo)} imagens de copo, {len(imgs_fone)} imagens de fone")
    
    # Se não conseguimos classificar automaticamente, usamos as imagens disponíveis
    if len(imgs_copo) == 0 and len(imgs_fone) == 0:
        print("Não foi possível classificar as imagens automaticamente. Usando imagens disponíveis.")
        random.shuffle(test_imgs)
        selected_imgs = test_imgs[:4] if len(test_imgs) >= 4 else test_imgs
    else:
        # Selecionar 2 imagens de cada classe, se disponíveis
        random.shuffle(imgs_copo)
        random.shuffle(imgs_fone)
        selected_imgs = []
        
        # 2 exemplos de copo
        if len(imgs_copo) >= 2:
            selected_imgs.extend(imgs_copo[:2])
        elif len(imgs_copo) == 1:
            selected_imgs.append(imgs_copo[0])
            
        # 2 exemplos de fone
        if len(imgs_fone) >= 2:
            selected_imgs.extend(imgs_fone[:2])
        elif len(imgs_fone) == 1:
            selected_imgs.append(imgs_fone[0])
        
        # Se não tivermos 4 imagens, complementar com outras
        while len(selected_imgs) < 4 and len(test_imgs) > len(selected_imgs):
            for img in test_imgs:
                if img not in selected_imgs:
                    selected_imgs.append(img)
                    break
                if len(selected_imgs) == 4:
                    break
    
    # Analisar os exemplos selecionados
    for i, img_path in enumerate(selected_imgs):
        print(f"\n==== EXEMPLO {i+1} ====")
        comparar_previsoes(img_path, modelo_30ep, modelo_60ep)

# Análise de diferentes limiares de confiança
if len(test_imgs) > 0:
    print("\n==== EFEITO DO LIMIAR DE CONFIANÇA NAS DETECÇÕES ====")
    
    # Usar uma imagem que sabemos ter detecções
    img_path = test_imgs[0]
    for img_path in test_imgs:
        result = modelo_60ep(img_path, conf=0.25)
        if len(result[0].boxes) > 0:
            break
    
    limiares = [0.25, 0.50, 0.75, 0.90]
    fig, axes = plt.subplots(1, len(limiares), figsize=(20, 5))
    
    for i, conf in enumerate(limiares):
        results = modelo_60ep(img_path, conf=conf)
        axes[i].imshow(results[0].plot())
        axes[i].set_title(f'Limiar: {conf}', fontsize=12)
        axes[i].axis('off')
        
        # Contar detecções
        n_detections = len(results[0].boxes)
        axes[i].text(10, 30, f'{n_detections} objetos detectados', 
                    color='white', fontsize=10, backgroundcolor='black')
    
    plt.tight_layout()
    plt.suptitle('Efeito do Limiar de Confiança nas Detecções (Modelo 60 épocas)', fontsize=14)
    plt.subplots_adjust(top=0.85)
    plt.show()

# Medir tempo médio de inferência
def medir_tempo_inferencia(modelo, img_path, repetitions=10):
    """Mede o tempo médio de inferência para uma imagem."""
    img = Image.open(img_path)
    tempos = []
    
    for _ in range(repetitions):
        start_time = time.time()
        _ = modelo(img)
        end_time = time.time()
        tempos.append((end_time - start_time) * 1000)  # Converter para ms
    
    return np.mean(tempos), np.std(tempos)

if len(test_imgs) > 0:
    print("\n==== TEMPOS DE INFERÊNCIA ====")
    img_path = test_imgs[0]
    
    # Medir tempo para modelo de 30 épocas
    try:
        media_30ep, std_30ep = medir_tempo_inferencia(modelo_30ep, img_path)
        print(f"Modelo 30 épocas: {media_30ep:.2f} ± {std_30ep:.2f} ms")
    except Exception as e:
        print(f"Erro ao medir tempo para modelo de 30 épocas: {e}")
    
    # Medir tempo para modelo de 60 épocas
    try:
        media_60ep, std_60ep = medir_tempo_inferencia(modelo_60ep, img_path)
        print(f"Modelo 60 épocas: {media_60ep:.2f} ± {std_60ep:.2f} ms")
    except Exception as e:
        print(f"Erro ao medir tempo para modelo de 60 épocas: {e}")

## 7. Análise dos Resultados de Teste dos Modelos YOLO

## Distribuição de Confiança

Analisando os histogramas de distribuição de confiança (Imagem 1), observamos padrões reveladores sobre o comportamento dos dois modelos:

- **Modelo de 30 épocas**: Apresenta uma distribuição mais polarizada, com picos de frequência nas faixas de confiança 0.73 e 0.93, indicando uma tendência a gerar previsões em níveis específicos de confiança.

- **Modelo de 60 épocas**: Embora também apresente picos nas mesmas faixas (0.70 e 0.90), mostra uma distribuição ligeiramente mais equilibrada nas faixas intermediárias e superiores.

- **Limiar mínimo**: Notamos que o modelo de 30 épocas começa com detecções de confiança mais baixa (0.60) em comparação ao modelo de 60 épocas (0.65), sugerindo que o modelo treinado por mais tempo estabelece um "padrão mínimo" mais elevado para suas detecções.

Esses padrões de distribuição corroboram nossa análise anterior de que o modelo de 60 épocas apresenta, em geral, maior confiança em suas previsões.

## Confiança Média por Classe

O gráfico de confiança média por classe (Imagem 2) revela uma dinâmica interessante e inesperada:

- **Classe "copo"**: O modelo de 30 épocas demonstra confiança média superior (0.90) em comparação ao modelo de 60 épocas (0.80). Esta inversão da tendência geral sugere uma possível especialização do modelo de 30 épocas para esta classe específica.

- **Classe "headphones"**: Confirmando nossa observação qualitativa anterior, o modelo de 60 épocas apresenta confiança média significativamente superior (0.88) em relação ao modelo de 30 épocas (0.76) para esta classe.

Este comportamento diferenciado entre classes pode ser explicado por:
1. Características visuais distintas entre as classes, com possível favorecimento de diferentes arquiteturas de modelo
2. Variações na distribuição e quantidade de exemplos de cada classe no conjunto de treinamento
3. Potencial especialização do modelo de 30 épocas em características específicas da classe "copo"

## Número de Detecções por Classe

A análise do número de detecções por classe (Imagem 3) revela outro aspecto importante do desempenho dos modelos:

- **Classe "copo"**: O modelo de 30 épocas detectou mais instâncias (6) em comparação ao modelo de 60 épocas (5).

- **Classe "headphones"**: O modelo de 60 épocas detectou o mesmo número de instâncias (5) que o modelo de 30 épocas.

- **Total de detecções**: O modelo de 30 épocas identificou 11 objetos, enquanto o modelo de 60 épocas identificou 10 objetos.

Isso indica que, contrariamente às expectativas baseadas no recall superior do modelo de 60 épocas observado na validação, o modelo de 30 épocas demonstrou uma capacidade ligeiramente superior de detecção no conjunto de teste limitado que analisamos.

## Correlação com Resultados Anteriores

Estes resultados de teste apresentam algumas divergências interessantes em relação à análise de validação anterior:

1. Na validação, o modelo de 60 épocas demonstrou superioridade consistente em todas as métricas, incluindo recall (capacidade de detectar todos os objetos presentes).

2. No teste, observamos um desempenho mais equilibrado e, em alguns aspectos, favorável ao modelo de 30 épocas, especialmente para a classe "copo".

Essas divergências podem ser atribuídas a:

- **Tamanho do conjunto de teste**: O número limitado de imagens no teste pode não representar completamente a distribuição do mundo real.

- **Variabilidade entre classes**: As duas classes apresentam características visuais e desafios de detecção distintos, que podem favorecer diferentes configurações de modelo.


In [ ]:

# Configuração do ambiente
print("Avaliando o desempenho dos modelos no conjunto de teste...")

# Diretório das imagens e labels de teste
test_img_dir = '../dataset/test/images'
test_label_dir = '../dataset/test/labels'  # Se disponível

# Verificar se o diretório de teste existe
if not os.path.exists(test_img_dir):
    print(f"AVISO: Diretório de imagens de teste '{test_img_dir}' não encontrado!")
else:
    print(f"Diretório de imagens de teste: {test_img_dir}")
    test_imgs = glob.glob(os.path.join(test_img_dir, '*.jpg')) + glob.glob(os.path.join(test_img_dir, '*.png'))
    print(f"Total de imagens de teste: {len(test_imgs)}")

# Carregar os modelos treinados (assumindo que já foram carregados na célula anterior)
try:
    # Verificar se os modelos já estão carregados
    if 'modelo_30ep' not in locals() and 'modelo_60ep' not in locals():
        print("Carregando modelos...")
        modelo_30ep = YOLO(os.path.join(models_dir, 'model_30epochs.pt'))
        modelo_60ep = YOLO(os.path.join(models_dir, 'model_60epochs.pt'))
    print("Modelos carregados com sucesso.")
except Exception as e:
    print(f"Erro ao carregar modelos: {e}")

# Função para salvar imagens com bounding boxes
def salvar_predicoes(modelo, img_path, output_dir, conf=0.25):
    """
    Executa o modelo na imagem e salva o resultado com bounding boxes.
    """
    results = modelo(img_path, conf=conf)
    os.makedirs(output_dir, exist_ok=True)
    img_name = os.path.basename(img_path)
    output_path = os.path.join(output_dir, img_name)
    result_plot = results[0].plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(result_plot)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()
    return results

# Função para coletar métricas de detecção
def coletar_metricas_deteccao(results, img_path, classes):
    """
    Coleta métricas sobre as detecções em uma imagem.
    """
    deteccoes = []
    
    if len(results[0].boxes) == 0:
        # Sem detecções
        deteccoes.append({
            'imagem': os.path.basename(img_path),
            'classe': 'nenhuma',
            'confianca': 0,
            'num_objetos': 0
        })
    else:
        for i, box in enumerate(results[0].boxes):
            classe_idx = int(box.cls)
            classe = classes[classe_idx] if classe_idx < len(classes) else f"desconhecido_{classe_idx}"
            confianca = float(box.conf)
            deteccoes.append({
                'imagem': os.path.basename(img_path),
                'classe': classe,
                'confianca': confianca,
                'num_objetos': len(results[0].boxes)
            })
    
    return deteccoes

# Diretórios para salvar os resultados
results_dir_30ep = os.path.join('../results', 'test_30ep')
results_dir_60ep = os.path.join('../results', 'test_60ep')

# Criar diretórios para resultados
os.makedirs(results_dir_30ep, exist_ok=True)
os.makedirs(results_dir_60ep, exist_ok=True)

# Executar os modelos em todas as imagens de teste
if 'test_imgs' in locals() and len(test_imgs) > 0:
    print(f"Processando {len(test_imgs)} imagens de teste...")
    
    # Listas para armazenar métricas
    metricas_30ep = []
    metricas_60ep = []
    
    # Limiar de confiança
    conf_threshold = 0.25
    
    for i, img_path in enumerate(test_imgs):
        print(f"Processando imagem {i+1}/{len(test_imgs)}: {os.path.basename(img_path)}")
        
        # Modelo de 30 épocas
        results_30ep = salvar_predicoes(modelo_30ep, img_path, results_dir_30ep, conf=conf_threshold)
        metricas_30ep.extend(coletar_metricas_deteccao(results_30ep, img_path, classes))
        
        # Modelo de 60 épocas
        results_60ep = salvar_predicoes(modelo_60ep, img_path, results_dir_60ep, conf=conf_threshold)
        metricas_60ep.extend(coletar_metricas_deteccao(results_60ep, img_path, classes))
    
    # Converter para DataFrame
    df_30ep = pd.DataFrame(metricas_30ep)
    df_60ep = pd.DataFrame(metricas_60ep)
    
    # Adicionar coluna identificando o modelo
    df_30ep['modelo'] = '30 épocas'
    df_60ep['modelo'] = '60 épocas'
    

    df_combined = pd.concat([df_30ep, df_60ep], ignore_index=True)
    
    # Salvar métricas em CSV
    df_30ep.to_csv(os.path.join('../results', 'metricas_30ep.csv'), index=False)
    df_60ep.to_csv(os.path.join('../results', 'metricas_60ep.csv'), index=False)
    df_combined.to_csv(os.path.join('../results', 'metricas_combined.csv'), index=False)
    
    print("Resultados salvos com sucesso!")
    
    # Análise Estatística
    print("\n=== ANÁLISE ESTATÍSTICA DAS DETECÇÕES ===")
    
    # 1. Confiança média por classe para cada modelo
    print("\nConfiança média por classe:")
    conf_por_classe_30ep = df_30ep[df_30ep['classe'] != 'nenhuma'].groupby('classe')['confianca'].mean()
    conf_por_classe_60ep = df_60ep[df_60ep['classe'] != 'nenhuma'].groupby('classe')['confianca'].mean()
    
    conf_df = pd.DataFrame({
        '30 épocas': conf_por_classe_30ep,
        '60 épocas': conf_por_classe_60ep
    })
    print(conf_df)
    
    # 2. Quantidade de detecções por classe
    print("\nQuantidade de detecções por classe:")
    det_por_classe_30ep = df_30ep[df_30ep['classe'] != 'nenhuma'].groupby('classe').size()
    det_por_classe_60ep = df_60ep[df_60ep['classe'] != 'nenhuma'].groupby('classe').size()
    
    det_df = pd.DataFrame({
        '30 épocas': det_por_classe_30ep,
        '60 épocas': det_por_classe_60ep
    })
    print(det_df)
    
    # 3. Imagens sem detecções
    sem_det_30ep = len(df_30ep[df_30ep['classe'] == 'nenhuma'])
    sem_det_60ep = len(df_60ep[df_60ep['classe'] == 'nenhuma'])
    
    print(f"\nImagens sem detecções:")
    print(f"30 épocas: {sem_det_30ep} de {len(test_imgs)} ({sem_det_30ep/len(test_imgs)*100:.1f}%)")
    print(f"60 épocas: {sem_det_60ep} de {len(test_imgs)} ({sem_det_60ep/len(test_imgs)*100:.1f}%)")
    
    # Visualização dos Resultados
    
    # 1. Histograma de confiança para cada modelo
    plt.figure(figsize=(15, 6))
    
    plt.subplot(1, 2, 1)
    df_30ep[df_30ep['classe'] != 'nenhuma']['confianca'].hist(bins=20, alpha=0.7)
    plt.title('Distribuição de Confiança - 30 épocas')
    plt.xlabel('Confiança')
    plt.ylabel('Frequência')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    df_60ep[df_60ep['classe'] != 'nenhuma']['confianca'].hist(bins=20, alpha=0.7)
    plt.title('Distribuição de Confiança - 60 épocas')
    plt.xlabel('Confiança')
    plt.ylabel('Frequência')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join('../results', 'histograma_confianca.png'))
    plt.show()
    
    # 2. Confiança média por classe (barplot)
    plt.figure(figsize=(12, 6))
    conf_df.plot(kind='bar', rot=0)
    plt.title('Confiança Média por Classe', fontsize=14)
    plt.xlabel('Classe', fontsize=12)
    plt.ylabel('Confiança Média', fontsize=12)
    plt.ylim(0, 1)
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(title='Modelo')
    plt.tight_layout()
    plt.savefig(os.path.join('../results', 'confianca_media_por_classe.png'))
    plt.show()
    
    # 3. Detecções por classe (barplot)
    plt.figure(figsize=(12, 6))
    det_df.plot(kind='bar', rot=0)
    plt.title('Detecções por Classe', fontsize=14)
    plt.xlabel('Classe', fontsize=12)
    plt.ylabel('Número de Detecções', fontsize=12)
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(title='Modelo')
    plt.tight_layout()
    plt.savefig(os.path.join('../results', 'deteccoes_por_classe.png'))
    plt.show()
    
    # Identificação e visualização de casos interessantes
    
    # 1. Detecções com maior diferença de confiança entre os modelos
    print("\n=== CASOS INTERESSANTES ===")
    
    # Criar um dicionário para mapear imagens e classes
    img_class_map = {}
    for _, row in df_30ep.iterrows():
        if row['classe'] != 'nenhuma':
            key = (row['imagem'], row['classe'])
            img_class_map[key] = {'conf_30ep': row['confianca']}
    
    for _, row in df_60ep.iterrows():
        if row['classe'] != 'nenhuma':
            key = (row['imagem'], row['classe'])
            if key in img_class_map:
                img_class_map[key]['conf_60ep'] = row['confianca']
            else:
                img_class_map[key] = {'conf_60ep': row['confianca']}
    
    # Calcular diferenças de confiança
    differences = []
    for key, values in img_class_map.items():
        if 'conf_30ep' in values and 'conf_60ep' in values:
            img, cls = key
            conf_diff = values['conf_60ep'] - values['conf_30ep']
            differences.append({
                'imagem': img,
                'classe': cls,
                'conf_30ep': values['conf_30ep'],
                'conf_60ep': values['conf_60ep'],
                'diferenca': conf_diff
            })
    
    if differences:
        diff_df = pd.DataFrame(differences)
        
        # Os 3 casos com maior diferença positiva (60ep > 30ep)
        top_positive = diff_df.sort_values('diferenca', ascending=False).head(3)
        print("\nCasos onde o modelo de 60 épocas tem maior confiança:")
        print(top_positive)
        
        # Os 3 casos com maior diferença negativa (30ep > 60ep)
        top_negative = diff_df.sort_values('diferenca', ascending=True).head(3)
        print("\nCasos onde o modelo de 30 épocas tem maior confiança:")
        print(top_negative)
        
        # Visualizar esses casos
        def visualizar_caso_interessante(img_name, title):
            img_path = None
            for path in test_imgs:
                if os.path.basename(path) == img_name:
                    img_path = path
                    break
            
            if img_path:
                # Resultados dos dois modelos
                result_30ep_path = os.path.join(results_dir_30ep, img_name)
                result_60ep_path = os.path.join(results_dir_60ep, img_name)
                
                fig, axes = plt.subplots(1, 3, figsize=(18, 6))
                
                # Imagem original
                img = Image.open(img_path)
                axes[0].imshow(img)
                axes[0].set_title('Imagem Original', fontsize=12)
                axes[0].axis('off')
                
                # 30 épocas
                if os.path.exists(result_30ep_path):
                    result_img = Image.open(result_30ep_path)
                    axes[1].imshow(result_img)
                else:
                    axes[1].text(0.5, 0.5, 'Imagem não encontrada', ha='center', va='center')
                axes[1].set_title('Modelo 30 épocas', fontsize=12)
                axes[1].axis('off')
                
                # 60 épocas
                if os.path.exists(result_60ep_path):
                    result_img = Image.open(result_60ep_path)
                    axes[2].imshow(result_img)
                else:
                    axes[2].text(0.5, 0.5, 'Imagem não encontrada', ha='center', va='center')
                axes[2].set_title('Modelo 60 épocas', fontsize=12)
                axes[2].axis('off')
                
                plt.tight_layout()
                plt.suptitle(title, fontsize=14)
                plt.subplots_adjust(top=0.85)
                plt.show()
            else:
                print(f"Imagem {img_name} não encontrada no conjunto de teste.")
        
        # Visualizar casos com maior diferença positiva
        if not top_positive.empty:
            best_case = top_positive.iloc[0]
            visualizar_caso_interessante(
                best_case['imagem'],
                f"Caso com maior confiança no modelo de 60 épocas\n"
                f"Classe: {best_case['classe']}, Diferença: {best_case['diferenca']:.4f} "
                f"(30ep: {best_case['conf_30ep']:.4f}, 60ep: {best_case['conf_60ep']:.4f})"
            )
        
        # Visualizar casos com maior diferença negativa
        if not top_negative.empty:
            worst_case = top_negative.iloc[0]
            visualizar_caso_interessante(
                worst_case['imagem'],
                f"Caso com maior confiança no modelo de 30 épocas\n"
                f"Classe: {worst_case['classe']}, Diferença: {worst_case['diferenca']:.4f} "
                f"(30ep: {worst_case['conf_30ep']:.4f}, 60ep: {worst_case['conf_60ep']:.4f})"
            )
    
    # 4. Relatório Final de Desempenho
    print("\n=== RELATÓRIO FINAL DE DESEMPENHO ===")
    
    # Estatísticas gerais
    print("\nEstatísticas Gerais:")
    
    # Confiança média geral
    conf_media_30ep = df_30ep[df_30ep['classe'] != 'nenhuma']['confianca'].mean()
    conf_media_60ep = df_60ep[df_60ep['classe'] != 'nenhuma']['confianca'].mean()
    
    print(f"Confiança média (30 épocas): {conf_media_30ep:.4f}")
    print(f"Confiança média (60 épocas): {conf_media_60ep:.4f}")
    print(f"Melhoria na confiança: {(conf_media_60ep - conf_media_30ep) / conf_media_30ep * 100:.2f}%")
    
    # Total de detecções
    total_det_30ep = len(df_30ep[df_30ep['classe'] != 'nenhuma'])
    total_det_60ep = len(df_60ep[df_60ep['classe'] != 'nenhuma'])
    
    print(f"\nTotal de detecções (30 épocas): {total_det_30ep}")
    print(f"Total de detecções (60 épocas): {total_det_60ep}")
    
    # Conclusão
    print("\nConclusão:")
    if conf_media_60ep > conf_media_30ep:
        print(f"O modelo de 60 épocas apresenta confiança média {(conf_media_60ep - conf_media_30ep) / conf_media_30ep * 100:.2f}% maior que o modelo de 30 épocas.")
    else:
        print(f"O modelo de 30 épocas apresenta confiança média {(conf_media_30ep - conf_media_60ep) / conf_media_60ep * 100:.2f}% maior que o modelo de 60 épocas.")
    
    if total_det_60ep > total_det_30ep:
        print(f"O modelo de 60 épocas detectou {total_det_60ep - total_det_30ep} objetos a mais que o modelo de 30 épocas.")
    else:
        print(f"O modelo de 30 épocas detectou {total_det_30ep - total_det_60ep} objetos a mais que o modelo de 60 épocas.")
        
    if sem_det_60ep < sem_det_30ep:
        print(f"O modelo de 60 épocas teve {sem_det_30ep - sem_det_60ep} menos imagens sem detecções.")
    else:
        print(f"O modelo de 30 épocas teve {sem_det_60ep - sem_det_30ep} menos imagens sem detecções.")
    
else:
    print("Não há imagens de teste disponíveis para processamento.")